In [1]:
import os
import json
import shutil
import time

from pyspark.sql import SparkSession
from pyspark import StorageLevel

1. Crear sesión Spark

In [2]:
spark = SparkSession.builder \
    .appName("Demo_Optimizacion_RDD") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext
print("Spark listo:", spark.version)

# Limpieza de carpetas de salida si ya existen
for path in ["demo_out_csv", "demo_out_json", "demo_out_parquet", "demo_resultados_parquet"]:
    if os.path.exists(path):
        shutil.rmtree(path)

Spark listo: 4.0.2


2. Preparar dataset de prueba

In [3]:
# Simularemos eventos por categoría
data = [
    ("ventas", "A", 100),
    ("ventas", "A", 120),
    ("ventas", "B", 90),
    ("marketing", "A", 40),
    ("marketing", "B", 60),
    ("marketing", "B", 30),
    ("soporte", "C", 20),
    ("soporte", "C", 25),
    ("soporte", "A", 15),
] * 10000  # repetir para que se note mejor

pairs = sc.parallelize(data) \
          .map(lambda x: (x[0], x[2]))  # (categoria, monto)

print("Ejemplo de datos:")
print(pairs.take(5))

Ejemplo de datos:
[('ventas', 100), ('ventas', 120), ('ventas', 90), ('marketing', 40), ('marketing', 60)]


3. Comparación groupByKey vs reduceByKey

In [4]:
# groupByKey
start = time.time()
group_result = pairs.groupByKey().mapValues(lambda vals: sum(vals)).collect()
group_time = time.time() - start

# reduceByKey
start = time.time()
reduce_result = pairs.reduceByKey(lambda a, b: a + b).collect()
reduce_time = time.time() - start

print("Resultado groupByKey:", group_result)
print("Tiempo groupByKey:", round(group_time, 4), "seg")

print("Resultado reduceByKey:", reduce_result)
print("Tiempo reduceByKey:", round(reduce_time, 4), "seg")


Resultado groupByKey: [('marketing', 1300000), ('ventas', 3100000), ('soporte', 600000)]
Tiempo groupByKey: 1.8584 seg
Resultado reduceByKey: [('marketing', 1300000), ('ventas', 3100000), ('soporte', 600000)]
Tiempo reduceByKey: 0.9924 seg


4. Broadcast variable

In [5]:
# Diccionario de referencia que queremos compartir con todos los nodos
categorias_desc = {
    "ventas": "Área comercial",
    "marketing": "Área de difusión",
    "soporte": "Área de atención"
}

bc_categorias = sc.broadcast(categorias_desc)

enriquecido = sc.parallelize(data).map(
    lambda x: (x[0], bc_categorias.value.get(x[0], "Sin categoría"), x[1], x[2])
)

print("Datos enriquecidos con broadcast:")
print(enriquecido.take(5))

Datos enriquecidos con broadcast:
[('ventas', 'Área comercial', 'A', 100), ('ventas', 'Área comercial', 'A', 120), ('ventas', 'Área comercial', 'B', 90), ('marketing', 'Área de difusión', 'A', 40), ('marketing', 'Área de difusión', 'B', 60)]


5. Acumulador

In [6]:
accum_altos = sc.accumulator(0)

def marcar_altos(reg):
    global accum_altos
    if reg[2] >= 100:
        accum_altos += 1
    return reg

procesado = sc.parallelize(data).map(marcar_altos)

# Acción para disparar ejecución
procesado.count()

print("Cantidad de registros con monto >= 100:", accum_altos.value)


Cantidad de registros con monto >= 100: 20000


6. Particionamiento y persistencia opcional

In [7]:
pair_rdd = sc.parallelize([
    ("a", 1), ("b", 1), ("a", 1), ("c", 1), ("b", 1), ("a", 1)
])

particionado = pair_rdd.partitionBy(4, lambda key: hash(key) % 4)
particionado.persist(StorageLevel.MEMORY_ONLY)

print("Número de particiones:", particionado.getNumPartitions())
print("Contenido:", particionado.collect())

Número de particiones: 4
Contenido: [('b', 1), ('c', 1), ('b', 1), ('a', 1), ('a', 1), ('a', 1)]


7. Crear DataFrame y guardar en CSV / JSON / Parquet

In [8]:
df = spark.createDataFrame(data, ["area", "subarea", "monto"])

df.write.mode("overwrite").option("header", True).csv("demo_out_csv")
df.write.mode("overwrite").json("demo_out_json")
df.write.mode("overwrite").parquet("demo_out_parquet")

print("Archivos escritos correctamente.")

Archivos escritos correctamente.


8. Leer datos desde CSV / JSON / Parquet

In [9]:
df_csv = spark.read.option("header", True).option("inferSchema", True).csv("demo_out_csv")
df_json = spark.read.json("demo_out_json")
df_parquet = spark.read.parquet("demo_out_parquet")

print("CSV:")
df_csv.show(3)

print("JSON:")
df_json.show(3)

print("Parquet:")
df_parquet.show(3)

CSV:
+------+-------+-----+
|  area|subarea|monto|
+------+-------+-----+
|ventas|      A|  100|
|ventas|      A|  120|
|ventas|      B|   90|
+------+-------+-----+
only showing top 3 rows
JSON:
+------+-----+-------+
|  area|monto|subarea|
+------+-----+-------+
|ventas|  100|      A|
|ventas|  120|      A|
|ventas|   90|      B|
+------+-----+-------+
only showing top 3 rows
Parquet:
+------+-------+-----+
|  area|subarea|monto|
+------+-------+-----+
|ventas|      A|  100|
|ventas|      A|  120|
|ventas|      B|   90|
+------+-------+-----+
only showing top 3 rows


9. Escribir resultado optimizado en formato columnar

In [10]:
resultado_df = df.groupBy("area").sum("monto") \
    .withColumnRenamed("sum(monto)", "total_monto")

resultado_df.show()

resultado_df.write.mode("overwrite").parquet("demo_resultados_parquet")
print("Resultado guardado en demo_resultados_parquet")

+---------+-----------+
|     area|total_monto|
+---------+-----------+
|  soporte|     600000|
|   ventas|    3100000|
|marketing|    1300000|
+---------+-----------+

Resultado guardado en demo_resultados_parquet


10. Cierre

In [11]:
spark.stop()
